In [ ]:
# adam_allen_cahn_periodic_stable.py
# ------------------------------------------------------------
# Pure-JAX Adam baseline for 1D Allen-Cahn from allen_cahn_python.mat
#
# PDE:
#     u_t - eps*u_xx + a*u^3 - a*u = 0
# equivalent:
#     u_t = eps*u_xx + a*u - a*u^3
#
# IC:
#     u(x,0) = x^2 cos(pi x)
#
# Periodic BC:
#     u(-1,t)   = u(1,t)
#     u_x(-1,t) = u_x(1,t)
#
# Adam baseline:
#   - resample every iteration
#   - PDE: 9000 interior points
#   - BC:  2000 time samples, paired left/right, value + derivative periodic BC
#   - IC:  1000 x samples at t=0
#   - same residual scaling as your AC SQP: r_scaled = r / max(1,a)
#   - input normalization ON, same as your AC SQP
#   - network [2, 30, 30, 30, 1], same as your AC SQP
#
# Saves:
#   allen_cahn_adam_theta.npy
#   hist_ac_adam_*.npy
#   allen_cahn_adam_heatmaps.png/pdf
# ------------------------------------------------------------

import os
import math
import time
from pathlib import Path

# Set device before importing JAX in a real script.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import scipy.io
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

import jax.numpy as jnp
from jax import random, grad, vmap, hessian

DTYPE = jnp.float64
EPS = DTYPE(1e-12)

# ============================================================
# Problem setup: match your Allen-Cahn data and AC-SQP setting
# ============================================================
X_MIN = -1.0
X_MAX = 1.0
T_MIN = 0.0
T_MAX = 1.0

EPS_AC = 1.0e-3
A_AC = 5.0
SCALE_RESIDUAL_BY_A = True

MAT_PATHS = [
    "allen_cahn_python.mat",
    "data/allen_cahn_python.mat",
    "allen_cahn.mat",
    "data/allen_cahn.mat",
]

# ============================================================
# Adam batch sizes requested by you
# ============================================================
B_PDE = 9000
B_BC = 2000
B_IC = 1000

# Loss weights. AC-SQP has W_PDE_OBJ=10 for objective.
# For Adam comparison, you can use W_PDE=10.0 to match SQP's PDE objective weight,
# or W_PDE=1.0 for unweighted component comparison.
W_PDE = 10.0
W_BCV = 1.0
W_BCD = 1.0
W_IC = 1.0

# Network: match AC-SQP
HIDDEN_DIM = 50
NUM_HIDDEN = 4

# Training
SEED = 0
N_STEPS = 50_000
LR = 1.0e-4
GRAD_CLIP = 2.0
PRINT_EVERY = 100
EVAL_EVERY = 1000
SAVE_PREFIX = "allen_cahn_adam"

# ============================================================
# Data loading
# ============================================================
def load_allen_cahn_mat(paths=MAT_PATHS):
    chosen = None
    for p in paths:
        if Path(p).exists():
            chosen = p
            break
    if chosen is None:
        raise FileNotFoundError(
            "Could not find Allen-Cahn data. Put allen_cahn_python.mat in current directory "
            "or data/allen_cahn_python.mat."
        )

    d = scipy.io.loadmat(chosen)
    t = np.asarray(d["t"]).squeeze().astype(np.float64)
    x = np.asarray(d["x"]).squeeze().astype(np.float64)
    usol = np.asarray(d["usol"], dtype=np.float64)

    # Your generator saves usol as (nt, nx+1). If transposed, fix it.
    if usol.shape == (x.size, t.size):
        usol = usol.T

    if usol.shape != (t.size, x.size):
        raise ValueError(f"Unexpected usol shape {usol.shape}; expected {(t.size, x.size)}")

    eps = float(np.asarray(d["eps"]).squeeze()) if "eps" in d else EPS_AC
    a = float(np.asarray(d["a"]).squeeze()) if "a" in d else A_AC
    return t, x, usol, eps, a, chosen


def make_grid_points(x_np, t_np):
    TT, XX = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([XX.reshape(-1), TT.reshape(-1)], axis=1)
    return X_grid, TT.shape

# ============================================================
# MLP utilities
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params


def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    return jnp.concatenate(flat_parts).astype(DTYPE), tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx:idx + W_size].reshape(W_shape)
        idx += W_size
        b = theta[idx:idx + b_size].reshape(b_shape)
        idx += b_size
        params.append({"W": W, "b": b})
    return params


def normalize_xt(X):
    # Same normalization as your AC-SQP file.
    x = X[:, 0:1]
    t = X[:, 1:2]
    x_n = DTYPE(2.0) * (x - DTYPE(X_MIN)) / (DTYPE(X_MAX) - DTYPE(X_MIN)) - DTYPE(1.0)
    t_n = DTYPE(2.0) * (t - DTYPE(T_MIN)) / (DTYPE(T_MAX) - DTYPE(T_MIN)) - DTYPE(1.0)
    return jnp.concatenate([x_n, t_n], axis=1)


def mlp_apply(params, X):
    h = normalize_xt(X)
    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h

# ============================================================
# PDE residual, BC, IC
# ============================================================
def pde_residual_raw(params, X, eps_ac=EPS_AC, a_ac=A_AC):
    """Raw Allen-Cahn residual: r = u_t - eps*u_xx + a*u^3 - a*u."""
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    H = vmap(hessian(u_fun))(X)

    u_t = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t - DTYPE(eps_ac) * u_xx + DTYPE(a_ac) * u ** 3 - DTYPE(a_ac) * u


def pde_residual(params, X, eps_ac=EPS_AC, a_ac=A_AC):
    r = pde_residual_raw(params, X, eps_ac, a_ac)
    if SCALE_RESIDUAL_BY_A:
        return r / jnp.maximum(DTYPE(1.0), DTYPE(a_ac))
    return r


@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]
    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux


def allen_cahn_ic_target(x):
    return x ** 2 * jnp.cos(jnp.pi * x)

# ============================================================
# Sampling: resample every Adam step
# ============================================================
def _sample_uniform(key, shape, lo, hi):
    return random.uniform(key, shape, minval=DTYPE(lo), maxval=DTYPE(hi), dtype=DTYPE)


def sample_pde_uniform(key, x_min, x_max, t_min, t_max, B=B_PDE):
    kx, kt = random.split(key, 2)
    x = _sample_uniform(kx, (B, 1), x_min, x_max)
    t = _sample_uniform(kt, (B, 1), t_min, t_max)
    return jnp.concatenate([x, t], axis=1)


def sample_bc_periodic(key, x_min, x_max, t_min, t_max, B=B_BC):
    t = _sample_uniform(key, (B, 1), t_min, t_max)
    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(t), t], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(t), t], axis=1)
    return XL, XR


def sample_ic(key, x_min, x_max, t0, B=B_IC):
    x = _sample_uniform(key, (B, 1), x_min, x_max)
    t = DTYPE(t0) * jnp.ones_like(x)
    Xic = jnp.concatenate([x, t], axis=1)
    u0 = allen_cahn_ic_target(x[:, 0])
    return Xic, u0

# ============================================================
# Adam loss
# ============================================================
def adam_loss_parts(theta, shapes, key,
                    x_min, x_max, t_min, t_max,
                    eps_ac, a_ac):
    """
    total = W_PDE*Lpde + W_BCV*Lbc_value + W_BCD*Lbc_derivative + W_IC*Lic
    Lpde uses the scaled residual, matching AC-SQP residual scaling.
    """
    params = unflatten_params(theta, shapes)
    key, k_pde, k_bc, k_ic = random.split(key, 4)

    # PDE residual points
    Xpde = sample_pde_uniform(k_pde, x_min, x_max, t_min, t_max, B=B_PDE)
    r_pde = pde_residual(params, Xpde, eps_ac, a_ac)
    Lpde = jnp.mean(r_pde ** 2)

    # periodic BC: value and derivative
    XL, XR = sample_bc_periodic(k_bc, x_min, x_max, t_min, t_max, B=B_BC)
    uL, uxL = u_and_ux(params, XL)
    uR, uxR = u_and_ux(params, XR)
    Lbc_val = jnp.mean((uL - uR) ** 2)
    Lbc_der = jnp.mean((uxL - uxR) ** 2)

    # IC at t=0
    Xic, u0 = sample_ic(k_ic, x_min, x_max, t_min, B=B_IC)
    u_ic = mlp_apply(params, Xic)[:, 0]
    Lic = jnp.mean((u_ic - u0) ** 2)

    total = (
        DTYPE(W_PDE) * Lpde
        + DTYPE(W_BCV) * Lbc_val
        + DTYPE(W_BCD) * Lbc_der
        + DTYPE(W_IC) * Lic
    )
    return total, (Lpde, Lbc_val, Lbc_der, Lic)

# ============================================================
# Adam optimizer
# ============================================================
@jax.jit
def adam_update(theta, g, m, v, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1.0 - b1) * g
    v = b2 * v + (1.0 - b2) * (g * g)
    t = t + 1
    mhat = m / (1.0 - b1 ** t)
    vhat = v / (1.0 - b2 ** t)
    theta = theta - lr * mhat / (jnp.sqrt(vhat) + eps)
    return theta, m, v, t


def train_adam(theta0, shapes,
               x_min, x_max, t_min, t_max,
               eps_ac, a_ac,
               x_np, t_np, usol_np,
               seed=SEED,
               n_steps=N_STEPS,
               lr=LR,
               grad_clip=GRAD_CLIP,
               b1=0.9,
               b2=0.999,
               adam_eps=1e-8,
               print_every=PRINT_EVERY,
               eval_every=EVAL_EVERY):
    key = random.PRNGKey(seed)
    theta = theta0
    m = jnp.zeros_like(theta)
    v = jnp.zeros_like(theta)
    t_adam = jnp.array(0, dtype=jnp.int32)

    hist = {
        "total": [],
        "pde": [],
        "bc_val": [],
        "bc_der": [],
        "ic": [],
        "gnorm": [],
        "rel_l2": [],
        "mse": [],
        "step": [],
    }

    best_rel = float("inf")
    best_theta = np.asarray(theta)

    @jax.jit
    def step(theta, m, v, t_adam, key):
        (loss_val, parts), g = jax.value_and_grad(
            lambda th: adam_loss_parts(
                th, shapes, key,
                x_min, x_max, t_min, t_max,
                eps_ac, a_ac,
            ),
            has_aux=True,
        )(theta)

        gnorm = jnp.linalg.norm(g)
        scale = jnp.minimum(DTYPE(1.0), DTYPE(grad_clip) / (gnorm + DTYPE(1e-12)))
        g = g * scale

        theta, m, v, t_adam = adam_update(theta, g, m, v, t_adam, lr, b1=b1, b2=b2, eps=adam_eps)
        return theta, m, v, t_adam, loss_val, parts, gnorm

    print("Adam Allen-Cahn baseline")
    print(f"PDE: u_t - {eps_ac:g} u_xx + {a_ac:g} u^3 - {a_ac:g} u = 0")
    print(f"Domain: x in [{x_min}, {x_max}], t in [{t_min}, {t_max}]")
    print(f"Batches: PDE={B_PDE}, BC={B_BC}, IC={B_IC}")
    print(f"Network: [2] + [{HIDDEN_DIM}]*{NUM_HIDDEN} + [1]")
    print(f"Weights: W_PDE={W_PDE}, W_BCV={W_BCV}, W_BCD={W_BCD}, W_IC={W_IC}")
    print(f"lr={lr:.1e}, grad_clip={grad_clip}, n_steps={n_steps}")
    print("Residual scaling by max(1,a):", SCALE_RESIDUAL_BY_A)

    t0_wall = time.time()

    for k in range(1, n_steps + 1):
        key, kstep = random.split(key, 2)
        theta, m, v, t_adam, total, (Lpde, Lbcv, Lbcd, Lic), gnorm = step(theta, m, v, t_adam, kstep)

        hist["total"].append(float(total))
        hist["pde"].append(float(Lpde))
        hist["bc_val"].append(float(Lbcv))
        hist["bc_der"].append(float(Lbcd))
        hist["ic"].append(float(Lic))
        hist["gnorm"].append(float(gnorm))

        do_eval = (k % eval_every == 0) or (k == 1) or (k == n_steps)
        rel_l2 = np.nan
        mse = np.nan
        if do_eval:
            mse, rel_l2, _ = eval_full_grid(theta, shapes, x_np, t_np, usol_np)
            hist["step"].append(k)
            hist["mse"].append(mse)
            hist["rel_l2"].append(rel_l2)
            if rel_l2 < best_rel:
                best_rel = rel_l2
                best_theta = np.asarray(theta)
                np.save(f"{SAVE_PREFIX}_best_theta.npy", best_theta)

        if k % print_every == 0 or k == 1:
            msg = (
                f"[adam k={k:6d}] total={float(total):.3e} "
                f"pde={float(Lpde):.3e} bcv={float(Lbcv):.3e} "
                f"bcd={float(Lbcd):.3e} ic={float(Lic):.3e} "
                f"| gnorm={float(gnorm):.2e}"
            )
            if do_eval:
                msg += f" | relL2={rel_l2:.3e} best={best_rel:.3e}"
            print(msg)

        if not np.isfinite(float(total)):
            print("Stopping early: non-finite loss.")
            break

    print(f"[adam done] elapsed={time.time() - t0_wall:.2f}s")
    print(f"[best] relL2={best_rel:.3e}")
    return theta, best_theta, hist

# ============================================================
# Evaluation and plotting
# ============================================================
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]


def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.asarray(X_grid_np, dtype=DTYPE)
    u_pred = np.asarray(predict_u(theta, shapes, X_grid)).reshape(grid_shape)
    u_true = np.asarray(usol_np, dtype=np.float64)
    mse = float(np.mean((u_pred - u_true) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12))
    return mse, rel_l2, u_pred


def save_heatmaps(x_np, t_np, u_true, u_pred, prefix=SAVE_PREFIX):
    err = np.abs(u_pred - u_true)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    vmin = min(float(np.min(u_true)), float(np.min(u_pred)))
    vmax = max(float(np.max(u_true)), float(np.max(u_pred)))

    panels = [
        (u_true, "Exact", vmin, vmax),
        (u_pred, "Adam prediction", vmin, vmax),
        (err, "Absolute error", None, None),
    ]

    for ax, (arr, title, lo, hi) in zip(axes, panels):
        kwargs = {}
        if lo is not None and hi is not None:
            kwargs["vmin"] = lo
            kwargs["vmax"] = hi
        im = ax.imshow(
            arr.T,
            origin="lower",
            aspect="auto",
            extent=[t_np[0], t_np[-1], x_np[0], x_np[-1]],
            interpolation="nearest",
            **kwargs,
        )
        ax.set_xlabel("t")
        ax.set_ylabel("x")
        ax.set_title(title)
        fig.colorbar(im, ax=ax)

    out_png = f"{prefix}_heatmaps.png"
    out_pdf = f"{prefix}_heatmaps.pdf"
    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    print("Saved:", out_png)
    print("Saved:", out_pdf)

    if os.environ.get("PLOT", "0") == "1":
        plt.show()
    else:
        plt.close(fig)

# ============================================================
# Main
# ============================================================
def main():
    t_np, x_np, usol_np, eps_file, a_file, data_path = load_allen_cahn_mat()
    print("Loaded Allen-Cahn reference from", data_path)
    print("Allen-Cahn data:", "t", t_np.shape, "x", x_np.shape, "usol", usol_np.shape)

    eps_ac = float(EPS_AC)
    a_ac = float(A_AC)
    if abs(eps_file - eps_ac) > 1e-14:
        print(f"WARNING: file eps={eps_file}, code EPS_AC={eps_ac}. Using code EPS_AC={eps_ac}.")
    if abs(a_file - a_ac) > 1e-14:
        print(f"WARNING: file a={a_file}, code A_AC={a_ac}. Using code A_AC={a_ac}.")

    x_min, x_max = float(x_np.min()), float(x_np.max())
    t_min, t_max = float(t_np.min()), float(t_np.max())

    layer_sizes = [2] + [HIDDEN_DIM] * NUM_HIDDEN + [1]
    key = random.PRNGKey(SEED)
    params0 = init_mlp_params(key, layer_sizes)
    theta0, shapes = flatten_params(params0)
    print("n_params =", int(theta0.size))

    theta_final, theta_best_np, hist = train_adam(
        theta0, shapes,
        x_min, x_max, t_min, t_max,
        eps_ac, a_ac,
        x_np, t_np, usol_np,
        seed=SEED,
        n_steps=N_STEPS,
        lr=LR,
        grad_clip=GRAD_CLIP,
        print_every=PRINT_EVERY,
        eval_every=EVAL_EVERY,
    )

    # Save history and final model
    np.save(f"{SAVE_PREFIX}_theta.npy", np.asarray(theta_final, dtype=np.float64))
    np.save(f"hist_ac_adam_total.npy", np.asarray(hist["total"]))
    np.save(f"hist_ac_adam_pde.npy", np.asarray(hist["pde"]))
    np.save(f"hist_ac_adam_bcv.npy", np.asarray(hist["bc_val"]))
    np.save(f"hist_ac_adam_bcd.npy", np.asarray(hist["bc_der"]))
    np.save(f"hist_ac_adam_ic.npy", np.asarray(hist["ic"]))
    np.save(f"hist_ac_adam_gnorm.npy", np.asarray(hist["gnorm"]))
    np.save(f"hist_ac_adam_eval_step.npy", np.asarray(hist["step"]))
    np.save(f"hist_ac_adam_eval_mse.npy", np.asarray(hist["mse"]))
    np.save(f"hist_ac_adam_eval_rel_l2.npy", np.asarray(hist["rel_l2"]))
    print("Saved model and hist_ac_adam_*.npy")

    # Final and best eval
    mse_f, rel_f, u_pred_f = eval_full_grid(theta_final, shapes, x_np, t_np, usol_np)
    print(f"[ADAM FINAL EVAL] full-grid MSE={mse_f:.6e}, relL2={rel_f:.6e}")

    theta_best = jnp.asarray(theta_best_np, dtype=DTYPE)
    mse_b, rel_b, u_pred_b = eval_full_grid(theta_best, shapes, x_np, t_np, usol_np)
    print(f"[ADAM BEST EVAL]  full-grid MSE={mse_b:.6e}, relL2={rel_b:.6e}")

    save_heatmaps(x_np, t_np, usol_np, u_pred_b, prefix=SAVE_PREFIX + "_best")


if __name__ == "__main__":
    main()


Loaded Allen-Cahn reference from data/allen_cahn.mat
Allen-Cahn data: t (201,) x (512,) usol (201, 512)
n_params = 7851
Adam Allen-Cahn baseline
PDE: u_t - 0.001 u_xx + 5 u^3 - 5 u = 0
Domain: x in [-1.0, 1.0], t in [0.0, 1.0]
Batches: PDE=9000, BC=2000, IC=1000
Network: [2] + [50]*4 + [1]
Weights: W_PDE=10.0, W_BCV=1.0, W_BCD=1.0, W_IC=1.0
lr=1.0e-04, grad_clip=2.0, n_steps=50000
Residual scaling by max(1,a): True
[adam k=     1] total=4.139e+00 pde=1.867e-01 bcv=1.167e+00 bcd=5.929e-01 ic=5.125e-01 | gnorm=4.82e+01 | relL2=1.278e+00 best=1.278e+00
[adam k=   100] total=1.279e-01 pde=1.847e-03 bcv=3.499e-04 bcd=8.315e-04 ic=1.083e-01 | gnorm=2.92e-01
[adam k=   200] total=1.239e-01 pde=1.666e-03 bcv=3.017e-05 bcd=1.394e-04 ic=1.071e-01 | gnorm=1.55e-01
[adam k=   300] total=1.235e-01 pde=1.558e-03 bcv=2.022e-05 bcd=1.613e-04 ic=1.077e-01 | gnorm=5.81e-01
[adam k=   400] total=1.120e-01 pde=1.855e-03 bcv=7.980e-06 bcd=1.360e-04 ic=9.333e-02 | gnorm=5.55e-01
[adam k=   500] total=1.208e